In [ ]:
import os
import numpy as np
import torch
from torch import nn
import unicodedata
import torch.nn.functional as F
from google.colab import drive

In [ ]:
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/char_rnn'
os.makedirs(DRIVE_DIR, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install -q pypdf
import re, unicodedata
from pypdf import PdfReader

pdf_path = '/content/drive/MyDrive/char_rnn/sach.pdf'   # file PDF của bạn
txt_path = '/content/drive/MyDrive/char_rnn/sach.txt'

reader = PdfReader(pdf_path)
text = '\n'.join(p.extract_text() or '' for p in reader.pages)

text = unicodedata.normalize('NFC', text)                 # chuẩn hóa dấu tiếng Việt
text = re.sub(r'^\s*\d+\s*$', '', text, flags=re.M)     # bỏ dòng chỉ có số trang
text = re.sub(r'-\n(\w)', r'\1', text)                  # nối từ bị ngắt gạch nối cuối dòng
text = re.sub(r'\n{3,}', '\n\n', text)                  # gộp dòng trống thừa

with open(txt_path, 'w', encoding='utf-8') as f:
    f.write(text)
print(len(text), 'ký tự')
print(text[:500])


470121 ký tự

Mục lục
Giới Thiệu
Chương 1: Thư Hồi Âm Để Trong Hộp Nhận Sữa - (1)
(2)
(3)
(4)
(5)
(6)
Chương 2: Thổi Kèn Harmonica Giữa Đêm Khuya - (1)
(2)
(3)
(4)
(5)
(6)
(7)
(8)
(9)
(10)
Chương 3: Ở Trong Xe Civic Đến Sáng - (1)
(2)
(3)
(4)
(5)
(6)
(7)
Chương 4: Mặc Niệm Bằng Nhạc Beatles - (1)
(2)
(3)
(4)
(5)
(6)
(7)
(8)
(9)
(10)
Chương 5: Cầu Nguyện Từ Trên Trời Cao - (1)
(2)
(3)
(4)
(5)
(6)
(7)
(8)
(9)
(10)
(11) - End
ĐIỀU KỲ DIỆU CỦA TIỆM TẠP
HÓA NAMIYA
Higashino Keigo
www.dtv-ebook.com
Giới Thiệu
Higa


In [ ]:
DATA_PATH = os.path.join(DRIVE_DIR, '.txt')
CKPT_PATH = os.path.join(DRIVE_DIR, 'char_rnn_last.pt')   # phiên train gần nhất
BEST_PATH = os.path.join(DRIVE_DIR, 'char_rnn_best.pt')   # model tốt nhất
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

Device: cuda


In [ ]:
# Sửa lại đường dẫn file đúng để tải dữ liệu
DATA_PATH = os.path.join(DRIVE_DIR, 'sach.txt')

with open(DATA_PATH, 'r', encoding='utf-8') as f:
    text = f.read()
text = unicodedata.normalize('NFC', text)

In [ ]:
# text[:100] #check file input -> print 0-100 characters

In [ ]:
# we create two dictionaries:
# 1. int2char, which maps integers to characters
# 2. char2int, which maps characters to unique integers
chars = tuple(sorted(set(text))) #set(text) loại bỏ kí tự trùng lặp trong từng từ
print('Số ký tự:', len(text), '| Từ điển:', len(chars))
int2char = dict(enumerate(chars)) #mã hóa số -> ký tự
char2int = {ch: ii for ii, ch in int2char.items()} #giải mã kỹ tự - số

# encode the text
encoded = np.array([char2int[ch] for ch in text])

Số ký tự: 470121 | Từ điển: 170


In [ ]:
# encoded[:100]

In [ ]:
#biến mỗi số nguyên thành vector one-hot ( vector toàn số 0 chỉ có chỉ số 1 tại vị trí chính )
def one_hot_encode(arr, n_labels):

    # Initialize the the encoded array
    one_hot = np.zeros((arr.size, n_labels), dtype=np.float32)

    # Fill the appropriate elements with ones
    one_hot[np.arange(one_hot.shape[0]), arr.flatten()] = 1.

    # Finally reshape it to get back to the original array
    one_hot = one_hot.reshape((*arr.shape, n_labels))

    return one_hot

In [ ]:
# check that the function works as expected
test_seq = np.array([[3, 5, 1]])
one_hot = one_hot_encode(test_seq, 8)

print(one_hot)

[[[0. 0. 0. 1. 0. 0. 0. 0.]
  [0. 0. 0. 0. 0. 1. 0. 0.]
  [0. 1. 0. 0. 0. 0. 0. 0.]]]


In [ ]:
#chia minibath
def get_batches(arr, batch_size, seq_length):
    batch_size_total = batch_size * seq_length
    # total number of batches we can make

    n_batches = len(arr)//batch_size_total
    # Keep only enough characters to make full batches
    arr = arr[:n_batches * batch_size_total]
    # Reshape into batch_size rows
    arr = arr.reshape((batch_size, -1))
    # iterate through the array, one sequence at a time
    for n in range(0, arr.shape[1], seq_length):
        # The features
        x = arr[:, n:n+seq_length]
        # The targets, shifted by one
        y = np.zeros_like(x)
        try:
            y[:, :-1], y[:, -1] = x[:, 1:], arr[:, n+seq_length]
        except IndexError:
            y[:, :-1], y[:, -1] = x[:, 1:], arr[:, 0]
        yield x, y

In [ ]:
#cách sửa trường hợp exception

def get_batches(arr, batch_size, seq_length):
    batch_size_total = batch_size * seq_length
    n_batches = (len(arr) - 1) // batch_size_total   # chừa 1 ký tự cho y
    x_all = arr[:n_batches*batch_size_total].reshape(batch_size, -1)
    y_all = arr[1:n_batches*batch_size_total+1].reshape(batch_size, -1)   # dịch 1
    for n in range(0, x_all.shape[1], seq_length):
        yield x_all[:, n:n+seq_length], y_all[:, n:n+seq_length]


In [ ]:
# arr = [[ 0  1  2  3  4  5  6  7  8]
#        [ 9 10 11 12 13 14 15 16 17]]      shape (2, 9),  seq_length = 3

In [ ]:
# #Ô kiểm thử: lấy batch đầu tiên và in góc 10×10 để xem y có lệch 1 so với x không
# batches = get_batches(encoded, 8, 50)
# x, y = next(batches)
# # printing out the first 10 items in a sequence
# print('x\n', x[:10, :10])
# print('\ny\n', y[:10, :10])

In [ ]:
# #Tạo lớp LSTM, phần lõi của mô hình
# self.lstm = nn.LSTM(input_size, n_hidden, n_layers,
#                             dropout=drop_prob, batch_first=True)

input_size, n_hidden, n_layers, drop_prob = 83, 512, 2, 0.5
lstm = nn.LSTM(input_size, n_hidden, n_layers,
               dropout=drop_prob, batch_first=True)
print(lstm)

LSTM(83, 512, num_layers=2, batch_first=True, dropout=0.5)


In [ ]:
# # #Tạo hidden state h và cell state c bằng 0 trước khi đọc
# # self.init_hidden()

# net = CharRNN(chars, n_hidden=512, n_layers=2)
# h = net.init_hidden(128)
# print(h[0].shape, h[1].shape)   # torch.Size([2, 128, 512]) x 2

In [ ]:
# train_on_gpu = torch.cuda.is_available()
# # if(train_on_gpu):
# #     print('Training on GPU!')
# # else:
# #     print('No GPU available, training on CPU; consider making n_epochs very small.')

In [ ]:
train_on_gpu = torch.cuda.is_available()
print('GPU:', train_on_gpu)

GPU: True


In [ ]:
class CharRNN(nn.Module):

    def __init__(self, tokens, n_hidden=256, n_layers=2,
                               drop_prob=0.5, lr=0.001):
        super().__init__()
        self.drop_prob = drop_prob
        self.n_layers = n_layers
        self.n_hidden = n_hidden
        self.lr = lr

        # creating character dictionaries
        self.chars = tokens
        self.int2char = dict(enumerate(self.chars))
        self.char2int = {ch: ii for ii, ch in self.int2char.items()}

        ## TODO: define the LSTM
        self.lstm = nn.LSTM(len(self.chars), n_hidden, n_layers,
                            dropout=drop_prob, batch_first=True)

        ## TODO: define a dropout layer
        self.dropout = nn.Dropout(drop_prob)

        ## TODO: define the final, fully-connected output layer
        self.fc = nn.Linear(n_hidden, len(self.chars))


    def forward(self, x, hidden):
        # ''' Forward pass through the network.
        #     These inputs are x, and the hidden/cell state `hidden`. '''

        ## TODO: Get the outputs and the new hidden state from the lstm
        r_output, hidden = self.lstm(x, hidden)

        ## TODO: pass through a dropout layer
        out = self.dropout(r_output)

        # Stack up LSTM outputs using view
        # you may need to use contiguous to reshape the output
        out = out.contiguous().view(-1, self.n_hidden)

        ## TODO: put x through the fully-connected layer
        out = self.fc(out)

        # return the final output and the hidden state
        return out, hidden


    def init_hidden(self, batch_size):
        ''' Initializes hidden state '''
        # Create two new tensors with sizes n_layers x batch_size x n_hidden,
        # initialized to zero, for hidden state and cell state of LSTM
        weight = next(self.parameters()).data

        if (train_on_gpu):
            hidden = (weight.new(self.n_layers, batch_size, self.n_hidden).zero_().cuda(),
                  weight.new(self.n_layers, batch_size, self.n_hidden).zero_().cuda())
        else:
            hidden = (weight.new(self.n_layers, batch_size, self.n_hidden).zero_(),
                      weight.new(self.n_layers, batch_size, self.n_hidden).zero_())

        return hidden


In [ ]:
def train(net, data, epochs=10, batch_size=10, seq_length=50, lr=0.001, clip=5, val_frac=0.1, print_every=10):
    # ''' Training a network

    #     Arguments
    #     ---------

    #     net: CharRNN network
    #     data: text data to train the network
    #     epochs: Number of epochs to train
    #     batch_size: Number of mini-sequences per mini-batch, aka batch size
    #     seq_length: Number of character steps per mini-batch
    #     lr: learning rate
    #     clip: gradient clipping
    #     val_frac: Fraction of data to hold out for validation
    #     print_every: Number of steps for printing training and validation loss

    # '''
    net.train()

    opt = torch.optim.Adam(net.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    # create training and validation data
    val_idx = int(len(data)*(1-val_frac))
    data, val_data = data[:val_idx], data[val_idx:]

    if(train_on_gpu):
        net.cuda()

    counter = 0
    n_chars = len(net.chars)
    for e in range(epochs):
        # initialize hidden state
        #Đầu mỗi epoch, xóa trắng bộ nhớ (h, c) về 0, vì đọc lại sách từ đầu
        h = net.init_hidden(batch_size)

        for x, y in get_batches(data, batch_size, seq_length):
            counter += 1

            # One-hot encode our data and make them Torch tensors
            x = one_hot_encode(x, n_chars)
            inputs, targets = torch.from_numpy(x), torch.from_numpy(y)

            if(train_on_gpu):
                inputs, targets = inputs.cuda(), targets.cuda()

            # Creating new variables for the hidden state, otherwise
            # we'd backprop through the entire training history
            h = tuple([each.data for each in h])

            # zero accumulated gradients
            net.zero_grad()

            # get the output from the model
            output, h = net(inputs, h)

            # calculate the loss and perform backprop
            loss = criterion(output, targets.view(-1))
            loss.backward()
            # `clip_grad_norm` helps prevent the exploding gradient problem in RNNs / LSTMs.
            nn.utils.clip_grad_norm_(net.parameters(), clip)
            opt.step()

            # loss stats
            if counter % print_every == 0:
                # Get validation loss
                val_h = net.init_hidden(batch_size)
                val_losses = []
                net.eval()
                for x, y in get_batches(val_data, batch_size, seq_length):
                    # One-hot encode our data and make them Torch tensors
                    x = one_hot_encode(x, n_chars)
                    x, y = torch.from_numpy(x), torch.from_numpy(y)

                    # Creating new variables for the hidden state, otherwise
                    # we'd backprop through the entire training history
                    val_h = tuple([each.data for each in val_h])

                    inputs, targets = x, y
                    if(train_on_gpu):
                        inputs, targets = inputs.cuda(), targets.cuda()

                    output, val_h = net(inputs, val_h)
                    val_loss = criterion(output, targets.view(batch_size*seq_length))

                    val_losses.append(val_loss.item())

                net.train() # reset to train mode after iterationg through validation data

                print("Epoch: {}/{}...".format(e+1, epochs),
                      "Step: {}...".format(counter),
                      "Loss: {:.4f}...".format(loss.item()),
                      "Val Loss: {:.4f}".format(np.mean(val_losses)))

In [ ]:
# define and print the net
n_hidden=512
n_layers=2

net = CharRNN(chars, n_hidden, n_layers)
print(net)

CharRNN(
  (lstm): LSTM(170, 512, num_layers=2, batch_first=True, dropout=0.5)
  (dropout): Dropout(p=0.5, inplace=False)
  (fc): Linear(in_features=512, out_features=170, bias=True)
)


In [ ]:
batch_size = 128
seq_length = 100
n_epochs = 20 # start smaller if you are just testing initial behavior

# train the model
train(net, encoded, epochs=n_epochs, batch_size=batch_size, seq_length=seq_length, lr=0.001, print_every=10)

Epoch: 1/20... Step: 10... Loss: 3.7117... Val Loss: 3.6930
Epoch: 1/20... Step: 20... Loss: 3.6658... Val Loss: 3.5874
Epoch: 1/20... Step: 30... Loss: 3.6524... Val Loss: 3.5866
Epoch: 2/20... Step: 40... Loss: 3.6588... Val Loss: 3.5836
Epoch: 2/20... Step: 50... Loss: 3.6132... Val Loss: 3.5758
Epoch: 2/20... Step: 60... Loss: 3.5962... Val Loss: 3.5728
Epoch: 3/20... Step: 70... Loss: 3.6465... Val Loss: 3.5672
Epoch: 3/20... Step: 80... Loss: 3.5607... Val Loss: 3.5418
Epoch: 3/20... Step: 90... Loss: 3.5048... Val Loss: 3.4733
Epoch: 4/20... Step: 100... Loss: 3.4648... Val Loss: 3.3499
Epoch: 4/20... Step: 110... Loss: 3.2728... Val Loss: 3.2180
Epoch: 4/20... Step: 120... Loss: 3.0918... Val Loss: 3.0138
Epoch: 4/20... Step: 130... Loss: 2.9210... Val Loss: 2.8861
Epoch: 5/20... Step: 140... Loss: 2.8725... Val Loss: 2.7846
Epoch: 5/20... Step: 150... Loss: 2.7687... Val Loss: 2.6751
Epoch: 5/20... Step: 160... Loss: 2.6437... Val Loss: 2.5752
Epoch: 6/20... Step: 170... Loss:

In [ ]:
model_name = 'rnn_20_epoch.net'

checkpoint = {'n_hidden': net.n_hidden,
              'n_layers': net.n_layers,
              'drop_prob': net.drop_prob,
              'state_dict': net.state_dict(),
              'tokens': net.chars}

with open(model_name, 'wb') as f:
    torch.save(checkpoint, f)

In [ ]:
def predict(net, char, h=None, top_k=None,temperature=0.8):
        # ''' Given a character, predict the next character.
        #     Returns the predicted character and the hidden state.
        # '''

        # tensor inputs
        x = np.array([[net.char2int[char]]])
        x = one_hot_encode(x, len(net.chars))
        x = one_hot_encode(np.array([[net.char2int[char]]]), len(net.chars))
        inputs = torch.from_numpy(x).to(device)

        if(train_on_gpu):
            inputs = inputs.cuda()

        # detach hidden state from history
        h = tuple([each.data for each in h])
        # get the output of the model
        out, h = net(inputs, h)

        # get the character probabilities
        # p = F.softmax(out, dim=1).data
        # p = F.softmax(out / temperature, dim=1).cpu()
        p = F.softmax(out / temperature, dim=1).detach().cpu()

        if(train_on_gpu):
            p = p.cpu() # move to cpu

        # get top characters
        if top_k is None:
            top_ch = np.arange(len(net.chars))
        else:
            p, top_ch = p.topk(top_k)
            top_ch = top_ch.numpy().squeeze()

        # select the likely next character with some element of randomness
        p = p.numpy().squeeze()
        char = np.random.choice(top_ch, p=p/p.sum())

        # return the encoded value of the predicted char and the hidden state
        return net.int2char[char], h

In [ ]:
#sample này dừng sinh văn bản khi gặp dấu . nhưng có các trường hợp bị ngắt giữa các tên như mrs. mr. dr. cung bị tính là kết thúc câu

In [ ]:
def sample(net, max_len=1000, prime='The', top_k=None,
           end_chars='.!?', min_len=20):
    if(train_on_gpu):
        net.cuda()
    else:
        net.cpu()

    net.eval() # eval mode

    # Chạy qua các ký tự mồi để tích lũy hidden state
    chars = [ch for ch in prime]
    h = net.init_hidden(1)
    for ch in prime:
        char, h = predict(net, ch, h, top_k=top_k)
    chars.append(char)

    n_new = 1                                  # số ký tự đã sinh
    while n_new < max_len:
        char, h = predict(net, chars[-1], h, top_k=top_k)
        chars.append(char)
        n_new += 1

        if n_new < min_len:                    # chưa đủ dài thì chưa xét dừng
            continue
        # 1) Hết đoạn văn: hai lần xuống dòng liên tiếp
        if char == '\n' and chars[-2] == '\n':
            break
        # 2) Hết câu: dấu câu rồi đến khoảng trắng hoặc xuống dòng  ->  ". "  ".\n"
        if chars[-2] in end_chars and char in ' \n':
            break
        # 3) Hết câu trong hội thoại: dấu câu, ngoặc kép, rồi khoảng trắng  ->  '." '
        if len(chars) >= 3 and chars[-3] in end_chars \
                and chars[-2] in '"\'' and char in ' \n':
            break

    return ''.join(chars).rstrip()             # bỏ khoảng trắng/xuống dòng thừa ở cuối

In [ ]:
# print(sample(net, 1000, prime='Anna', top_k=5))

In [ ]:
with open('rnn_20_epoch.net', 'rb') as f:
    checkpoint = torch.load(f)

loaded = CharRNN(checkpoint['tokens'], n_hidden=checkpoint['n_hidden'], n_layers=checkpoint['n_layers'], drop_prob=checkpoint['drop_prob'])
loaded.load_state_dict(checkpoint['state_dict'])

<All keys matched successfully>

In [ ]:
print(sample(loaded, 2000, top_k=5, prime="Higashino"))

NameError: name 'sample' is not defined